In [ ]:
# =========================
# RQ3: Effect of Preprocessing
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

try:
    from imblearn.over_sampling import SMOTE
    smote_available = True
except:
    smote_available = False
    print("SMOTE not available. Full pipeline will run without SMOTE.")

# -------------------------
# 1. Load Data
# -------------------------
df = pd.read_csv(
    "/kaggle/input/datasets/sharmajicoder/gaming-and-mental-health/gaming_mental_health_10M_40features.csv"
)

df = df.sample(n=20000, random_state=42)

TARGET = df.columns[-1]

df = df.dropna(subset=[TARGET])

# -------------------------
# 2. Encode Target
# -------------------------
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
y = (y > 0).astype(int)

X_raw = df.drop(TARGET, axis=1)

# -------------------------
# 3. Train/Test Split
# -------------------------
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

results = []

# -------------------------
# Function for evaluation
# -------------------------
def evaluate_model(step_name, X_train, X_test, y_train, y_test):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Preprocessing Strategy": step_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1-score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

# -------------------------
# 4. Strategy 1: Raw Data
# Fill missing values simply + one-hot encoding
# -------------------------
X_train_raw1 = X_train_raw.fillna(0)
X_test_raw1 = X_test_raw.fillna(0)

X_train_raw1 = pd.get_dummies(X_train_raw1)
X_test_raw1 = pd.get_dummies(X_test_raw1)

X_train_raw1, X_test_raw1 = X_train_raw1.align(
    X_test_raw1,
    join="left",
    axis=1,
    fill_value=0
)

evaluate_model(
    "Raw Data",
    X_train_raw1,
    X_test_raw1,
    y_train,
    y_test
)

# -------------------------
# 5. Strategy 2: Missing Value Imputation
# -------------------------
X_train_imp = X_train_raw.copy()
X_test_imp = X_test_raw.copy()

numeric_cols = X_train_imp.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X_train_imp.select_dtypes(include=["object", "category", "bool"]).columns

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_imp[numeric_cols] = num_imputer.fit_transform(X_train_imp[numeric_cols])
X_test_imp[numeric_cols] = num_imputer.transform(X_test_imp[numeric_cols])

X_train_imp[categorical_cols] = cat_imputer.fit_transform(X_train_imp[categorical_cols])
X_test_imp[categorical_cols] = cat_imputer.transform(X_test_imp[categorical_cols])

X_train_imp = pd.get_dummies(X_train_imp)
X_test_imp = pd.get_dummies(X_test_imp)

X_train_imp, X_test_imp = X_train_imp.align(
    X_test_imp,
    join="left",
    axis=1,
    fill_value=0
)

evaluate_model(
    "Imputation",
    X_train_imp,
    X_test_imp,
    y_train,
    y_test
)

# -------------------------
# 6. Strategy 3: Scaling + Encoding
# -------------------------
scaler = StandardScaler()

X_train_scaled = X_train_imp.copy()
X_test_scaled = X_test_imp.copy()

X_train_scaled = scaler.fit_transform(X_train_scaled)
X_test_scaled = scaler.transform(X_test_scaled)

evaluate_model(
    "Scaling + Encoding",
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

# -------------------------
# 7. Strategy 4: Full Pipeline with SMOTE
# -------------------------
X_train_full = X_train_scaled.copy()
X_test_full = X_test_scaled.copy()

if smote_available:
    smote = SMOTE(random_state=42)
    X_train_full, y_train_full = smote.fit_resample(X_train_full, y_train)
else:
    y_train_full = y_train

evaluate_model(
    "Full Pipeline\n(SMOTE)",
    X_train_full,
    X_test_full,
    y_train_full,
    y_test
)

# -------------------------
# 8. Save Results Table
# -------------------------
table = pd.DataFrame(results)
table.to_csv("RQ3_table.csv", index=False)

print("\n=== RQ3 Results Table ===")
print(table)

# -------------------------
# 9. Plot Figure
# -------------------------
plt.figure(figsize=(9, 6))

metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
markers = ["o", "s", "^", "D"]

for i, metric in enumerate(metrics):
    plt.plot(
        table["Preprocessing Strategy"],
        table[metric],
        marker=markers[i],
        linewidth=2,
        label=metric
    )

    for j, value in enumerate(table[metric]):
        plt.text(
            j,
            value + 0.003,
            f"{value:.2f}",
            ha="center",
            fontsize=9
        )

plt.title("RQ3: Effect of Preprocessing", fontsize=14, fontweight="bold")
plt.xlabel("Preprocessing Strategy", fontsize=12, fontweight="bold")
plt.ylabel("Score", fontsize=12, fontweight="bold")

plt.ylim(
    max(0, table[metrics].min().min() - 0.05),
    min(1, table[metrics].max().max() + 0.05)
)

plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("RQ3_figure.pdf", bbox_inches="tight")
plt.show()